In [ ]:
%cd ..

In [ ]:
from pathlib import Path
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

from spesia_ner.metrics import compute_metrics
from spesia_ner.datasets import ClinicalRecordsDataset, DataCollatorForMultiLabelTokenClassification
from spesia_ner.trainers import MultiLabelTokenTrainer
from spesia_ner.trainers import EarlyStoppingCallback

model_id = "jhu-clsp/mmBERT-base"
dataset_path = Path("data/SemClinBr/annotated_records")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

semantic_groups_to_consider = [
    'Anatomy',
    'Chemicals & Drugs',
    'Concepts & Ideas',
    'Devices',
    'Disorders',
    'Living Beings',
    'Organizations',
    'Phenomena',
    'Physiology',
    'Procedures'
]

data_split_method = 'iterative'
label_type = 'semantic_groups'
annotation_scheme = 'IO'
additional_descriptors = 'oversampling'

train_dataset = ClinicalRecordsDataset(
    dataset_path, 
    tokenizer, 
    split="train",
    semantic_groups_to_consider=semantic_groups_to_consider,
    data_split_method=data_split_method,
    label_type=label_type,
    annotation_scheme=annotation_scheme,
    )

val_dataset = ClinicalRecordsDataset(
    dataset_path, 
    tokenizer, 
    split="val",
    semantic_groups_to_consider=semantic_groups_to_consider,
    data_split_method=data_split_method,
    label_type=label_type,
    annotation_scheme=annotation_scheme
    )

test_dataset = ClinicalRecordsDataset(
    dataset_path, 
    tokenizer, 
    split="test",
    semantic_groups_to_consider=semantic_groups_to_consider,
    data_split_method=data_split_method,
    label_type=label_type,
    annotation_scheme=annotation_scheme
    )

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=25,
    gradient_accumulation_steps=2,
    num_train_epochs=40,
    learning_rate=5e-5,
    include_for_metrics=['inputs'],
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1",
)

data_collator = DataCollatorForMultiLabelTokenClassification(
    pad_token_id=tokenizer.pad_token_id,
    max_length=512,
    num_labels=train_dataset.num_labels
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# supress warnings
import warnings
warnings.filterwarnings("ignore")

def iterate_training(model_id, training_args, train_dataset, val_dataset, data_collator, compute_metrics, num_iterations=10):
    test_metrics = []

    for j in range(num_iterations):
        model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=train_dataset.num_labels)
        trainer = MultiLabelTokenTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(patience=5)],
            # pos_weight=train_dataset.pos_weight.clamp(min=1)
        )
        trainer.train()
        trainer.eval_dataset = test_dataset
        test_iteration_results = trainer.evaluate()
        test_metrics.append(test_iteration_results)

        # Get predictions object
        pred_output = trainer.predict(test_dataset)


        # Get probabilities (sigmoid for multilabel)
        probs = 1 / (1 + np.exp(-pred_output.predictions))
        labels = pred_output.label_ids.astype(int)

        # For single-label: flatten arrays
        if probs.shape[-1] == 1:
            probs_flat = probs.reshape(-1)
            labels_flat = labels.reshape(-1)
        else:
            # For multilabel: plot for each label
            probs_flat = probs.reshape(-1, probs.shape[-1])
            labels_flat = labels.reshape(-1, labels.shape[-1])

        # Plot for each label (or just one if single-label)
        for i in range(probs_flat.shape[-1] if probs_flat.ndim > 1 else 1):
            if probs_flat.ndim > 1:
                y_score = probs_flat[:, i]
                y_true = labels_flat[:, i]
                label_name = f"Label {test_dataset.labels_to_consider[i]}"
            else:
                y_score = probs_flat
                y_true = labels_flat
                label_name = "Label"

            precision, recall, thresholds = precision_recall_curve(y_true, y_score)
            plt.plot(recall, precision, label=label_name)

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.subplots_adjust(right=0.7)  # make room on the right for the legend
        plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
        plt.savefig(f"precision_recall_curve_{data_split_method}_{annotation_scheme}_{additional_descriptors}_{j:02}.png", bbox_inches="tight")
        plt.show()

    return test_metrics

test_metrics = iterate_training(model_id, training_args, train_dataset, val_dataset, data_collator, compute_metrics)

In [ ]:
train_dataset.get_label_count(train_dataset.label_type)

In [ ]:
train_dataset.label_map

In [ ]:
train_dataset.pos_weight

In [ ]:
import pandas as pd

test_metrics = pd.DataFrame(test_metrics)

exp_name = f"{data_split_method}_{annotation_scheme}_{additional_descriptors}_report"

cols_of_interest = [
    "eval_loss",
    "eval_macro_precision",
    "eval_macro_recall",
    "eval_macro_f1",
    "eval_micro_precision",
    "eval_micro_recall",
    "eval_micro_f1",
]

test_metrics.agg(["mean", "std"])[cols_of_interest].to_markdown(f"{exp_name}.md")

In [ ]:
test_metrics.to_csv(f"{exp_name}.csv")